# Практическое занятие: статистическое описание событий кибербезопасности

**Рабочая тетрадь студента**  
Продолжительность: 80 минут. Работайте с базовой линией (`baseline`) и окном расследования (`investigation`). До задания 5 не используйте `is_malicious` и `attack_type` как подсказку.

## Правила интерпретации

1. Сначала определите тип признака и допустимые операции.
2. Статистический тест не заменяет график и содержательную интерпретацию.
3. Параметры теоретических распределений оценивайте только по базовой линии.
4. В выводе называйте наблюдаемый признак, направление изменения и возможный сценарий атаки.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

DATA_PATH = 'cyber_statistics_practice.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])

baseline = df[df['period'] == 'baseline'].copy()
investigation = df[df['period'] == 'investigation'].copy()

df.head()

## Задание 1. Типы признаков и шкалы измерения (3 балла)

1. Проверьте типы данных, пропуски и дубликаты.
2. Для указанных признаков определите: качественный/количественный; номинальный/порядковый либо дискретный/непрерывный.
3. Преобразуйте `user_role`, `request_method`, `destination_service` в `category`, а `severity_level` — в упорядоченную категорию `Low < Medium < High < Critical`. Объясните, почему `is_malicious`, несмотря на 0/1, является категориальным признаком.

In [ ]:
# 1.1: df.dtypes
# 1.2: заполните таблицу классификации признаков по типам
feature_types = pd.DataFrame({
    'feature': ['request_method', 'severity_level', 'requests_per_min',
                'response_time_ms', 'source_port', 'is_malicious'],
    'type_data': ['...', '...', '...', '...', '...', '...']
})
feature_types

In [ ]:
# 1.3: преобразование типов
severity_order = ['Low', 'Medium', 'High', 'Critical']
# df[...] = ...
df.dtypes

## Задание 2. Эмпирические распределения (4 балла)

По базовой линии:
- постройте ряд распределения `requests_per_min` с абсолютной частотой, PMF и ECDF;
- разбейте `response_time_ms` на 12 интервалов и постройте таблицу интервальных частот;
- постройте два графика: PMF/ECDF для дискретного признака и гистограмму для непрерывного;
- получите частотную таблицу `request_method`.

В выводе поясните различие PMF, PDF/гистограммы и ECDF.

In [ ]:
# 2.1: ряд распределения requests_per_min
# Подсказка: value_counts().sort_index(), normalize=True, cumsum()
request_dist = ...
request_dist.head()

In [ ]:
# 2.2: pd.cut(..., bins=12), value_counts(sort=False)
# 2.3: два подграфика matplotlib
# 2.4: частоты request_method

## Задание 3. Числовые характеристики (4 балла)

Для `cpu_load_percent`, `response_time_ms` и `payload_size_bytes` рассчитайте среднее, медиану, стандартное отклонение, MAD, Q1, Q3 и IQR отдельно для baseline и investigation.

Ответьте:
1. Для каких признаков среднее заметно сильнее медианы реагирует на окно расследования?
2. Почему MAD и IQR предпочтительнее стандартного отклонения при тяжёлых хвостах и выбросах?
3. Рассчитайте порог 95-го квантиля `requests_per_min` по baseline методом `higher`.

In [ ]:
# 3.1: напишите функцию descriptive_stats(series)
def descriptive_stats(series):
    # Верните Series со статистиками mean, median, std, MAD, Q1, Q3, IQR
    pass

# 3.2: примените функцию по period
# 3.3: порог q95 методом higher

## Задание 4. Оценивание законов распределения (5 баллов)

Используйте только baseline. Для каждого признака сопоставьте эмпирическое и теоретическое распределения:

- `cpu_load_percent` — нормальное: гистограмма + PDF, критерий Д’Агостино;
- `payload_size_bytes` — логнормальное: проверьте нормальность `ln(X)`;
- `response_time_ms` — экспоненциальное: оцените scale средним, выполните диагностический K–S тест;
- `requests_per_min` — Пуассона: оцените λ средним и визуально сравните PMF;
- `source_port` — равномерное: 32 бина, нормированная энтропия и χ² (ожидаемая частота каждого бина должна быть ≥ 5).

Сформируйте таблицу: признак, предполагаемый закон, оценённые параметры, p-value/энтропия, вывод. Не делайте вывод только по p-value.

In [ ]:
# 4.1: нормальное распределение CPU и normaltest
# 4.2: normaltest(np.log(payload_size_bytes))
# 4.3: kstest(response_time_ms, 'expon', args=(0, scale_hat))
# 4.4: эмпирическая PMF и stats.poisson.pmf
# 4.5: энтропия и chisquare по 32 бинам

## Задание 5. Статистическое расследование (4 балла)

До пункта 5.4 не используйте метки атак.

1. Сравните baseline и investigation по медиане и Q95 количественных признаков.
2. Сравните доли методов запросов и уровней критичности; для `request_method` рассчитайте моду, индекс вариации `VR = 1 - p_mode` и энтропию.
3. Сравните нормированную энтропию распределения `source_port` по 64 равным бинам.
4. Сформулируйте минимум три признака инцидента и предположите сценарии. Только затем раскройте `attack_type` и проверьте гипотезы группировкой.
5. Напишите итог SOC-аналитика в 4–6 предложениях: факт, статистическое свидетельство, гипотеза атаки, ограничение анализа.

In [ ]:
# 5.1: таблица median и q95 по period
# 5.2: распределения request_method и severity_level; mode, VR, entropy
# 5.3: энтропия source_port по 64 бинам

# 5.4: только после формулировки гипотезы сопоставить с известными атаками
# pd.crosstab(df['period'], df['attack_type'])
# df.groupby('attack_type')[...].median()

## Самопроверка и сдача

Сдайте ноутбук с выполненными ячейками, пятью краткими выводами и итоговой запиской SOC. Графики должны иметь заголовок, подписи осей и легенду там, где сравниваются две кривые.